In [ ]:
import numpy as np
import os
import mat73
import scipy
import matplotlib.pyplot as plt
from load_data_function import load_data,save_data,fig_plot,battery_soh_plot,smooth_soh

In [ ]:
"""
(a) 1-C cycles (current = 740mA): time (t, in seconds), voltage (v, in Volts), charge (q, in mAh) and temperature (T, in degrees Celsius),
(b) Pseudo-OCV cycles (current = 40mA): time (t, in seconds), voltage (v, in Volt), charge (q, in mAh) and temperature (T, in degree Celsius),
1-C and pseudo-OCV cycles were recorded every 100 cycles of drive cycles.
The drive cycles were based on an Urban Artemis driving profile.
"""


package_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data\Oxford Battery Degradation Dataset\Oxford_Battery_Degradation_Dataset_1.mat'
package_list=['package_1']
print(package_list)
package_data=scipy.io.loadmat(package_path)

Oxford_data={}
Oxford_SOH={}
for i,package in enumerate(package_list):
    print(f'package: {package}')
    battery_list=['Cell1','Cell2','Cell3','Cell4','Cell5','Cell6','Cell7','Cell8']
    print(f'battery_list: {battery_list}')

    package1={}
    package2={}
    for j,battery in enumerate(battery_list):
        print(f'battery: {battery}')
        battery_data=package_data[battery]
        #print(battery_data.dtype.names)
        cycle_list=battery_data.dtype.names
        voltage=[]
        current=[]
        time=[]
        capacity=[]
        package1[f'battery_{j+1}']=[]
        package2[f'battery_{j+1}']=[]
        for k,cycle in enumerate(cycle_list):
            cycle_data=battery_data[cycle][0,0]
            print(cycle)
            C1ch_data=cycle_data['C1ch'][0,0]
            C1dc_data=cycle_data['C1dc'][0,0]
            OCVch_data=cycle_data['OCVch'][0,0]
            OCVdc_data=cycle_data['OCVdc'][0,0]

            t=C1ch_data['t'][0,0]
            v=C1ch_data['v'][0,0]
            q=0.001*C1ch_data['q'][0,0]
            ii = np.gradient(q.flatten(), t.flatten())
            ii=np.expand_dims(ii,1)
            #print(q)
            #print(i.shape)
            #print(i)
            voltage=v.T
            #print(voltage.shape)
            current=ii.T
            time_segment = t.T
            #time=60 * time_segment  # 转换为秒
            time=time_segment

            Capacity_max=np.max(abs(q))
            #print(capacity)
            #print(Capacity_max)
            soh=Capacity_max/0.74
            #print(soh)
            package1[f'battery_{j+1}'].append(np.concatenate((voltage,current,time),axis=0))
            package2[f'battery_{j+1}'].append(soh)


    Oxford_data[f'package_{i+1}']=package1
    Oxford_SOH[f'package_{i+1}']=package2

In [ ]:
save_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data\Oxford_dataset'
#save_data(Oxford_data, os.path.join(save_path, 'Oxford_data.pkl'))
Oxford_data=load_data(os.path.join(save_path, 'Oxford_data.pkl'))
#save_data(Oxford_SOH, os.path.join(save_path, 'Oxford_SOH.pkl'))
Oxford_SOH=load_data(os.path.join(save_path, 'Oxford_SOH.pkl'))


In [ ]:
print(Oxford_data.keys())

In [ ]:
print(Oxford_data['package_1'].keys())


In [ ]:
print(Oxford_data['package_1']['battery_1'][0].shape)

In [ ]:
print(Oxford_SOH['package_1']['battery_1'][0]
      )

In [ ]:
fig_plot(Oxford_SOH['package_1']['battery_1'],Oxford_SOH['package_1']['battery_2'])

In [ ]:
package='package_1'
battery_soh_plot(Oxford_SOH,Oxford_SOH[package].keys(),package)

In [ ]:
smooth_Oxford_SOH=smooth_soh(Oxford_SOH,method='gaussian',sigma=1)
package='package_1'
battery_soh_plot(smooth_Oxford_SOH,smooth_Oxford_SOH[package].keys(),package)

In [ ]:
fig_plot(Oxford_data[package]['battery_8'][0][2])
print(Oxford_data[package]['battery_8'][0][2])